In [2]:
pip install pandas openpyxl 

Note: you may need to restart the kernel to use updated packages.


In [52]:
import pandas as pd

# Load the Dataset

In [53]:
# Read the Excel file
df_raw = pd.read_excel('Online Retail.xlsx')
df = df_raw

# Preview the data
print(df.shape)
print(df.head())
print(df.info())

(541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       ----------

# Preprocessing Data

## 1. Remove Cancellation Transactions

Finding: 9,288 rows have InvoiceNo starting with 'C'

Action: Remove them

Reason: As discussed earlier, cancellations don't represent real purchasing decisions. Including them would create false patterns.

In [54]:
# Check cancellations before removal
print("Before removal:", df.shape)
print("Cancellation rows:", df['InvoiceNo'].astype(str).str.startswith('C').sum())

# Remove cancellations
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Verify
print("After removal:", df.shape)

Before removal: (541909, 8)
Cancellation rows: 9288
After removal: (532621, 8)


## 2. Remove Negative & Zero Quantities

Finding: 10,624 rows have negative Quantity values (min = -80,995!)

Action: Remove rows where Quantity ≤ 0

Reason: Negative quantities represent returns or adjustments — not actual purchases. Zero quantities add no information to a basket.

In [55]:
# Check before removal
print("Before removal:", df.shape)
print("Negative or zero quantities:", (df['Quantity'] <= 0).sum())

# Remove negative and zero quantities
df = df[df['Quantity'] > 0]

# Verify
print("After removal:", df.shape)

Before removal: (532621, 8)
Negative or zero quantities: 1336
After removal: (531285, 8)


## 3. Remove Zero or Negative Unit Prices

Finding: 2,517 rows have UnitPrice ≤ 0

Action: Remove rows where UnitPrice ≤ 0

Reason: Free or negative-priced items likely represent samples, errors, or adjustments — not genuine retail transactions.

In [56]:
# Check before removal
print("Before removal:", df.shape)
print("Zero or negative prices:", (df['UnitPrice'] <= 0).sum())

# Remove zero and negative unit prices
df = df[df['UnitPrice'] > 0]

# Verify
print("After removal:", df.shape)

Before removal: (531285, 8)
Zero or negative prices: 1181
After removal: (530104, 8)


## 4. Remove Non-Standard Stock Codes

Finding: Stock codes like POST, DOT, BANK CHARGES, AMAZONFEE, M, D etc. — 33 non-standard codes found

Action: Keep only StockCodes that match the standard 5-digit format

Reason: These are service charges, postage fees, and admin entries — not actual products. Including them would generate meaningless rules like "customers who buy products also pay postage".


In [57]:
# Check before removal
print("Before removal:", df.shape)
print("Non-standard stock codes:", df[~df['StockCode'].astype(str).str.match(r'^\d{5}')]['StockCode'].unique())
print("Non-standard rows:", df[~df['StockCode'].astype(str).str.match(r'^\d{5}')].shape[0])

# Remove non-standard stock codes
df = df[df['StockCode'].astype(str).str.match(r'^\d{5}')]

# Verify
print("After removal:", df.shape)

Before removal: (530104, 8)
Non-standard stock codes: ['POST' 'C2' 'DOT' 'M' 'BANK CHARGES' 'AMAZONFEE' 'DCGS0076' 'DCGS0003'
 'gift_0001_40' 'DCGS0070' 'm' 'gift_0001_50' 'gift_0001_30'
 'gift_0001_20' 'DCGS0069' 'DCGSSBOY' 'DCGSSGIRL' 'gift_0001_10' 'S'
 'PADS' 'DCGS0004' 'B']
Non-standard rows: 2379
After removal: (527725, 8)


#### Note

Also notice valid stock codes like 85123A also pass this filter even though they have a letter at the end. That's intentional — the regex only checks that the code starts with 5 digits, which is the standard format for real products in this dataset.

## 5. Remove Duplicate Rows

Finding: 5,268 exact duplicate rows exist

Action: Drop duplicates

Reason: Duplicates could be data entry errors or system glitches. They would artificially inflate the frequency of certain items, distorting support values.


In [58]:
# Check before removal
print("Before removal:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Verify
print("After removal:", df.shape)


Before removal: (527725, 8)
Duplicate rows: 5221
After removal: (522504, 8)


## 6. Handle Missing CustomerID

Finding: 135,080 rows (24.93%) have no CustomerID

Action: Keep these rows — do NOT remove them

Reason: For MBA we only need InvoiceNo and StockCode/Description. CustomerID is not required to identify a basket. Removing 25% of the data would significantly reduce the dataset.

## 7. Handle Missing Descriptions

Finding: 1,454 rows have missing Description

Action: Use StockCode instead of Description for item identification

Reason: StockCode is always present and is a more reliable identifier. Since 650 StockCodes have multiple descriptions (due to typos/variations), StockCode is actually the cleaner choice for mining.

## 8. Country Filtering Decision

Finding: 91.43% of transactions are from the United Kingdom

Action: Filter to UK only (optional but recommended)

Reason: Since the vast majority of data is from the UK, including other countries may introduce noise from different purchasing cultures. Focusing on UK gives cleaner, more consistent patterns.

In [59]:
# Check before filtering
print("Before filtering:", df.shape)
print("\nCountry distribution:")
print(df['Country'].value_counts())

# Filter to UK only
df = df[df['Country'] == 'United Kingdom']

# Verify
print("\nAfter filtering:", df.shape)

Before filtering: (522504, 8)

Country distribution:
Country
United Kingdom          478838
Germany                   8643
France                    8085
EIRE                      7768
Spain                     2417
Netherlands               2322
Belgium                   1935
Switzerland               1927
Portugal                  1455
Australia                 1180
Norway                    1048
Channel Islands            743
Italy                      741
Finland                    647
Cyprus                     601
Unspecified                442
Sweden                     427
Austria                    384
Denmark                    367
Poland                     325
Japan                      321
Israel                     292
Hong Kong                  275
Singapore                  215
Iceland                    182
USA                        179
Canada                     150
Greece                     142
Malta                      109
United Arab Emirates        67
European 

## 9. Remove Single-Item Baskets

Finding: 5,841 invoices contain only one item

Action: Remove these after grouping

Reason: A basket with only one item can never produce an association rule — you need at least 2 items to find a relationship between them.

In [60]:
# Check before removal
print("Before removal:", df.shape)

# Count items per invoice
basket_counts = df.groupby('InvoiceNo')['StockCode'].count()
print("Single item baskets:", (basket_counts == 1).sum())
print("Total invoices:", basket_counts.shape[0])

# Get invoices with more than one item
multi_item_invoices = basket_counts[basket_counts > 1].index

# Keep only multi-item baskets
df = df[df['InvoiceNo'].isin(multi_item_invoices)]

# Verify
print("\nAfter removal:", df.shape)
print("Remaining invoices:", df['InvoiceNo'].nunique())

Before removal: (478838, 8)
Single item baskets: 1389
Total invoices: 17901

After removal: (477449, 8)
Remaining invoices: 16512


#### Why Remove Single Item Baskets?

| Reason | Explanation |
|---|---|
| No association possible | You need at least 2 items to form a rule like "A → B" |
| Waste of computation | Single item baskets add processing overhead with zero benefit |
| Cleaner results | Removes noise from the frequent itemset generation |

## 10. Attributes to Ignore & Why?

| Attribute | Decision | Reason |
|---|---|---| 
| Quantity | Ignore (binarize) | MBA only cares if an item was bought, not how many. Convert to 1/0 |
| InvoiceDate | Ignore | Not needed for basic MBA (could be used for seasonal analysis but outside scope) |
| UnitPrice | Ignore(after filtering) | Price is used only for cleaning, not in the basket matrix |
| CustomerID | Ignore | We group by Invoice, not customer |
| Country | Used for filtering only | Not included in basket |

# Exploratory Data Analysis

In [91]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ───────────────────────────────────────────────────────────
df = pd.read_excel('Online Retail.xlsx')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['StockCode']   = df['StockCode'].astype(str).str.strip()

# ── Apply preprocessing filters ────────────────────────────────────────────
df_clean = df.copy()
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]
df_clean = df_clean[df_clean['StockCode'].str.match(r'^\d{5}')]
df_clean = df_clean.drop_duplicates()
df_clean = df_clean[df_clean['Country'] == 'United Kingdom']
df_clean['Month']   = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['Hour']    = df_clean['InvoiceDate'].dt.hour
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

# ── Colour palette ─────────────────────────────────────────────────────────
DARK_BLUE  = '#1B3A5C'
MID_BLUE   = '#2E6DA4'
LIGHT_BLUE = '#D6E4F0'
GREEN      = '#1E8449'
ORANGE     = '#D35400'
GREY       = '#7F8C8D'

# ── Global style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
})

# ── Figure layout ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor('white')

fig.text(0.5, 0.98,
         'Online Retail Dataset — Exploratory Data Analysis',
         ha='center', va='top', fontsize=18,
         fontweight='bold', color=DARK_BLUE)
fig.text(0.5, 0.965,
         'UK Transactions | December 2010 – December 2011',
         ha='center', va='top', fontsize=11, color=GREY)

# ── Plot 1: Monthly Transaction Volume ────────────────────────────────────
ax1 = fig.add_subplot(3, 2, 1)

monthly = df_clean.groupby('Month')['InvoiceNo'].nunique().reset_index()
monthly['Month_str'] = monthly['Month'].astype(str).str[2:]

bars1 = ax1.bar(monthly['Month_str'], monthly['InvoiceNo'],
                color=MID_BLUE, edgecolor='white', linewidth=0.5)

# Highlight the peak month
peak_idx = monthly['InvoiceNo'].idxmax()
bars1[peak_idx].set_color(ORANGE)

# Add value labels on each bar
for bar, val in zip(bars1, monthly['InvoiceNo']):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 20,
             str(val), ha='center', va='bottom',
             fontsize=7, color=DARK_BLUE)

ax1.annotate('Peak: Nov 2011',
             xy=(peak_idx, monthly['InvoiceNo'].max()),
             xytext=(peak_idx - 3, monthly['InvoiceNo'].max() - 300),
             arrowprops=dict(arrowstyle='->', color=ORANGE),
             fontsize=8, color=ORANGE)

ax1.set_title('Monthly Transaction Volume',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax1.set_xlabel('Month', fontsize=9, color=GREY)
ax1.set_ylabel('Number of Invoices', fontsize=9, color=GREY)
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.tick_params(axis='y', labelsize=8)

# ── Plot 2: Top 10 Countries by Invoices ──────────────────────────────────
ax2 = fig.add_subplot(3, 2, 2)

# Use raw df (before UK filter) to show UK dominance
country_counts = (
    df[~df['InvoiceNo'].astype(str).str.startswith('C')]
    .groupby('Country')['InvoiceNo']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

colors_country = [DARK_BLUE if c == 'United Kingdom' else LIGHT_BLUE
                  for c in country_counts.index]

ax2.barh(country_counts.index[::-1],
         country_counts.values[::-1],
         color=colors_country[::-1],
         edgecolor='white')

for i, val in enumerate(country_counts.values[::-1]):
    ax2.text(val + 50, i,
             f'{val:,}', va='center',
             fontsize=8, color=DARK_BLUE)

ax2.set_title('Top 10 Countries by Number of Invoices',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax2.set_xlabel('Number of Invoices', fontsize=9, color=GREY)
ax2.set_xlim(0, country_counts.max() * 1.15)
ax2.tick_params(labelsize=8)

# ── Plot 3: Top 10 Best-Selling Products ──────────────────────────────────
ax3 = fig.add_subplot(3, 2, 3)

top_products = (
    df_clean.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Truncate long product names for readability
labels = [p[:35] + '...' if len(p) > 35 else p
          for p in top_products.index[::-1]]

colors_prod = [ORANGE if i == 9 else MID_BLUE
               for i in range(10)]

ax3.barh(labels, top_products.values[::-1],
         color=colors_prod, edgecolor='white')

for i, val in enumerate(top_products.values[::-1]):
    ax3.text(val + 200, i,
             f'{val:,}', va='center',
             fontsize=8, color=DARK_BLUE)

ax3.set_title('Top 10 Best-Selling Products (by Quantity)',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax3.set_xlabel('Total Quantity Sold', fontsize=9, color=GREY)
ax3.set_xlim(0, top_products.max() * 1.18)
ax3.tick_params(labelsize=8)

# ── Plot 4: Basket Size Distribution ──────────────────────────────────────
ax4 = fig.add_subplot(3, 2, 4)

basket_size = df_clean.groupby('InvoiceNo')['StockCode'].nunique()
basket_capped = basket_size[basket_size <= 50]

ax4.hist(basket_capped, bins=30,
         color=MID_BLUE, edgecolor='white', linewidth=0.5)

ax4.axvline(basket_size.median(), color=ORANGE, linestyle='--',
            linewidth=1.5,
            label=f'Median: {basket_size.median():.0f} items')
ax4.axvline(basket_size.mean(), color=GREEN, linestyle='--',
            linewidth=1.5,
            label=f'Mean: {basket_size.mean():.1f} items')

ax4.set_title('Basket Size Distribution (≤ 50 items)',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax4.set_xlabel('Number of Distinct Products per Invoice',
               fontsize=9, color=GREY)
ax4.set_ylabel('Number of Invoices', fontsize=9, color=GREY)
ax4.legend(fontsize=8)
ax4.tick_params(labelsize=8)

# ── Plot 5: Monthly Revenue ────────────────────────────────────────────────
ax5 = fig.add_subplot(3, 2, 5)

monthly_rev = df_clean.groupby('Month')['Revenue'].sum().reset_index()
monthly_rev['Month_str'] = monthly_rev['Month'].astype(str).str[2:]

ax5.fill_between(range(len(monthly_rev)),
                 monthly_rev['Revenue'],
                 alpha=0.3, color=MID_BLUE)
ax5.plot(range(len(monthly_rev)),
         monthly_rev['Revenue'],
         color=DARK_BLUE, linewidth=2,
         marker='o', markersize=5)

peak_rev_idx = monthly_rev['Revenue'].idxmax()
ax5.annotate(
    f'Peak\n£{monthly_rev["Revenue"].max() / 1000:.0f}K',
    xy=(peak_rev_idx, monthly_rev['Revenue'].max()),
    xytext=(peak_rev_idx - 2.5, monthly_rev['Revenue'].max() * 0.90),
    arrowprops=dict(arrowstyle='->', color=ORANGE),
    fontsize=8, color=ORANGE
)

ax5.set_xticks(range(len(monthly_rev)))
ax5.set_xticklabels(monthly_rev['Month_str'], rotation=45, fontsize=8)
ax5.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'£{x / 1000:.0f}K'))

ax5.set_title('Monthly Revenue (UK Transactions)',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax5.set_xlabel('Month', fontsize=9, color=GREY)
ax5.set_ylabel('Revenue (GBP)', fontsize=9, color=GREY)
ax5.tick_params(axis='y', labelsize=8)

# ── Plot 6: Transaction Volume by Hour of Day ─────────────────────────────
ax6 = fig.add_subplot(3, 2, 6)

hourly = df_clean.groupby('Hour')['InvoiceNo'].nunique()

colors_hour = [ORANGE if h == hourly.idxmax() else MID_BLUE
               for h in hourly.index]

ax6.bar(hourly.index, hourly.values,
        color=colors_hour, edgecolor='white', linewidth=0.5)

ax6.annotate(
    f'Peak: {hourly.idxmax()}:00',
    xy=(hourly.idxmax(), hourly.max()),
    xytext=(hourly.idxmax() + 2, hourly.max() - 150),
    arrowprops=dict(arrowstyle='->', color=ORANGE),
    fontsize=8, color=ORANGE
)

ax6.set_title('Transaction Volume by Hour of Day',
              fontsize=12, fontweight='bold', color=DARK_BLUE, pad=10)
ax6.set_xlabel('Hour of Day', fontsize=9, color=GREY)
ax6.set_ylabel('Number of Invoices', fontsize=9, color=GREY)
ax6.set_xticks(range(0, 24))
ax6.tick_params(labelsize=8)

# ── Save ───────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('eda_plots.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("EDA plots saved successfully to 'eda_plots.png'")

EDA plots saved successfully to 'eda_plots.png'


![EDA_plots](images/eda_plots.png)

# Transforming the data into a basket matrix 

In [61]:
# Create the basket matrix
basket = df.groupby(['InvoiceNo', 'StockCode'])['Quantity'].sum().unstack().fillna(0)

# Convert to binary (1 if item was bought, 0 if not)
basket = basket.map(lambda x: 1 if x > 0 else 0)

# Verify
print("Basket matrix shape:", basket.shape)
print("\nSample of basket matrix:")
print(basket.iloc[:5, :5])

Basket matrix shape: (16512, 3890)

Sample of basket matrix:
StockCode  10002  10080  10120  10125  10133
InvoiceNo                                   
536365         0      0      0      0      0
536366         0      0      0      0      0
536367         0      0      0      0      0
536368         0      0      0      0      0
536372         0      0      0      0      0


## Save the basket matrix to a CSV file

In [62]:
# Save the basket matrix to a CSV file
basket.to_csv('basket_matrix.csv')
print("Basket matrix saved successfully!")
print("Shape:", basket.shape)

Basket matrix saved successfully!
Shape: (16512, 3890)


## Summary of All Preprocessing Steps

| Step | Rows |
|---|---|
| Original dataset | 541,909 |
| After removing cancellations | 532,621 |
| After removing negative/zero quantities | 531,285 |
| After removing zero/negative prices | 530,104 |
| After removing non-standard stock codes | 527,725 |
| After removing duplicates | 522,504 |
| After filtering UK only | 478,838 |
| After removing single item baskets | 477,449 |
| **Final basket matrix** | **16,512 × 3,890**| 

# Running the Apriori algorithm with appropriate parameter settings

In [63]:
pip install mlxtend

/opt/homebrew/anaconda3/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=12978) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


## Load the basket matrix

In [64]:
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd
import time


# Load the basket matrix
basket = pd.read_csv('basket_matrix.csv', index_col=0)
basket = basket.astype(bool)

print("Basket matrix loaded successfully!")
print("Shape:", basket.shape)

Basket matrix loaded successfully!
Shape: (16512, 3890)


## Find frequent itemsets

In [65]:
# Run Apriori — Cycle 1: min_support = 0.01 Memorry Error

print(f"Apriori — Cycle 2")
print(f"==================")
# Run Apriori — Cycle 2
start = time.time()
frequent_itemsets = apriori(basket,
                            min_support=0.02,    # 2% — practical starting point
                            use_colnames=True,
                            max_len=None)
end = time.time()

print(f"Time taken: {round(end-start, 2)} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets)}")
print(frequent_itemsets.sort_values('support', ascending=False).head(10))

print(f"Apriori — Cycle 3")
print(f"==================")
# Run Apriori — Cycle 3
start = time.time()
frequent_itemsets_cycle3 = apriori(basket,
                            min_support=0.03,   # 3% - faster but fewer rules
                            use_colnames=True,
                            max_len=None)
end = time.time()

print(f"Time taken: {round(end-start, 2)} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets_cycle3)}")
print(frequent_itemsets_cycle3.sort_values('support', ascending=False).head(10))



Apriori — Cycle 2
Time taken: 1.72 seconds
Frequent itemsets found: 488
      support             itemsets
345  0.126332  frozenset({85123A})
342  0.116279  frozenset({85099B})
146  0.100836   frozenset({22423})
291  0.095567   frozenset({47566})
11   0.084302   frozenset({20725})
318  0.082909   frozenset({84879})
125  0.077095   frozenset({22197})
205  0.074673   frozenset({22720})
151  0.073765   frozenset({22457})
13   0.073522   frozenset({20727})
Apriori — Cycle 3
Time taken: 0.13 seconds
Frequent itemsets found: 181
      support             itemsets
154  0.126332  frozenset({85123A})
151  0.116279  frozenset({85099B})
66   0.100836   frozenset({22423})
125  0.095567   frozenset({47566})
8    0.084302   frozenset({20725})
142  0.082909   frozenset({84879})
55   0.077095   frozenset({22197})
83   0.074673   frozenset({22720})
67   0.073765   frozenset({22457})
10   0.073522   frozenset({20727})


## Generate association rules

In [66]:
# Step 2 — Generate association rules
rules = association_rules(frequent_itemsets,
                          metric="lift",
                          min_threshold=1.0)

print(f"Total rules generated: {len(rules)}")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


Total rules generated: 312
           antecedents          consequents   support  confidence      lift
0  frozenset({85099B})   frozenset({20711})  0.020470    0.176042  5.744664
1   frozenset({20711})  frozenset({85099B})  0.020470    0.667984  5.744664
2   frozenset({20712})   frozenset({21931})  0.025739    0.578231  8.449341
3   frozenset({21931})   frozenset({20712})  0.025739    0.376106  8.449341
4   frozenset({22386})   frozenset({20712})  0.024043    0.343129  7.708493
5   frozenset({20712})   frozenset({22386})  0.024043    0.540136  7.708493
6   frozenset({20712})   frozenset({22411})  0.022650    0.508844  7.422283
7   frozenset({22411})   frozenset({20712})  0.022650    0.330389  7.422283
8   frozenset({20712})  frozenset({85099B})  0.028706    0.644898  5.546122
9  frozenset({85099B})   frozenset({20712})  0.028706    0.246875  5.546122


In [67]:
print(f"Average support: {sum(rules.support)/len(rules)}")
print(f"Average confidence: {sum(rules.confidence)/len(rules)}")
print(f"Average lift: {sum(rules.lift)/len(rules)}")

Average support: 0.025413141646789903
Average confidence: 0.4392561979706997
Average lift: 7.6011082751577375


In [68]:
# # Get product descriptions from raw data
# df_raw = pd.read_excel('Online Retail.xlsx')
# desc_map = df_raw.dropna(subset=['Description']).groupby('StockCode')['Description'].first().to_dict()

# # Function to convert stock codes to descriptions
# def get_desc(itemset):
#     return ', '.join([str(desc_map.get(item, item)) for item in itemset])

# # Add description columns
# rules.insert(2, 'antecedent_desc', rules['antecedents'].apply(get_desc))
# rules.insert(3, 'consequent_desc', rules['consequents'].apply(get_desc))

# # Round metrics
# rules['support']    = rules['support'].round(4)
# rules['confidence'] = rules['confidence'].round(4)
# rules['lift']       = rules['lift'].round(4)

# # Preview
# print(rules[['antecedents','antecedent_desc',
#                          'consequents','consequent_desc',
#                          'support','confidence','lift']].head(10).to_string())

           antecedents          antecedent_desc          consequents          consequent_desc  support  confidence    lift
0  frozenset({85099B})  JUMBO BAG RED RETROSPOT   frozenset({20711})                    20711   0.0205      0.1760  5.7447
1   frozenset({20711})                    20711  frozenset({85099B})  JUMBO BAG RED RETROSPOT   0.0205      0.6680  5.7447
2   frozenset({20712})                    20712   frozenset({21931})                    21931   0.0257      0.5782  8.4493
3   frozenset({21931})                    21931   frozenset({20712})                    20712   0.0257      0.3761  8.4493
4   frozenset({22386})                    22386   frozenset({20712})                    20712   0.0240      0.3431  7.7085
5   frozenset({20712})                    20712   frozenset({22386})                    22386   0.0240      0.5401  7.7085
6   frozenset({20712})                    20712   frozenset({22411})                    22411   0.0227      0.5088  7.4223
7   frozenset({2

# Save Resuling Rules to a CSV file

In [78]:
# Create resulting_rules dataframe with selected columns
resulting_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
                          
# Sort by lift descending
resulting_rules = resulting_rules.sort_values('lift', ascending=False).reset_index(drop=True)

# Convert frozensets to readable strings for saving
resulting_rules['antecedents'] = resulting_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
resulting_rules['consequents'] = resulting_rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Save to CSV
resulting_rules.to_csv('resulting_rules.csv', index=False)

print(f"Total rules saved: {len(resulting_rules)}")
print("\nPreview:")
print(resulting_rules.head(10).to_string())


Total rules saved: 312

Preview:
    antecedents   consequents  support  confidence     lift
0        84596F        84596B   0.0203      0.8014  26.2046
1        84596B        84596F   0.0203      0.6634  26.2046
2         22577         22578   0.0223      0.7230  24.9748
3         22578         22577   0.0223      0.7699  24.9748
4  22699, 22697         22698   0.0298      0.7029  16.6269
5         22698  22699, 22697   0.0298      0.7049  16.6269
6         22866         22865   0.0200      0.6187  16.2673
7         22865         22866   0.0200      0.5271  16.2673
8         22697  22699, 22698   0.0298      0.5279  15.9938
9  22699, 22698         22697   0.0298      0.9028  15.9938


# FP-Growth Algorithm

In [88]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
import pandas as pd
import time

# Load basket
basket = pd.read_csv('basket_matrix.csv', index_col=0)
basket = basket.astype(bool)

# Run FP-Growth
print("Running FP-Growth with min_support=0.02...")
start = time.time()
frequent_itemsets = fpgrowth(basket,
                             min_support=0.02,   # Same threshold as Apriori
                             use_colnames=True,
                             max_len=None)
end = time.time()

print(f"Time taken: {round(end-start, 2)} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets)}")

# Generate rules — exactly the same as Apriori
rules = association_rules(frequent_itemsets,
                          metric='lift',
                          min_threshold=1.0)

print(f"Total rules generated: {len(rules)}")
print(rules[['antecedents','consequents','support','confidence','lift']].head(10))

Running FP-Growth with min_support=0.02...
Time taken: 1.11 seconds
Frequent itemsets found: 488
Total rules generated: 312
           antecedents          consequents   support  confidence      lift
0   frozenset({22961})   frozenset({22960})  0.025436    0.419580  6.941996
1   frozenset({22960})   frozenset({22961})  0.025436    0.420842  6.941996
2   frozenset({22960})   frozenset({22720})  0.020652    0.341683  4.575731
3   frozenset({22720})   frozenset({22960})  0.020652    0.276561  4.575731
4  frozenset({85123A})   frozenset({82482})  0.022166    0.175455  2.697505
5   frozenset({82482})  frozenset({85123A})  0.022166    0.340782  2.697505
6   frozenset({82482})  frozenset({82494L})  0.029918    0.459963  8.640393
7  frozenset({82494L})   frozenset({82482})  0.029918    0.562002  8.640393
8   frozenset({21733})  frozenset({85123A})  0.027980    0.659058  5.216862
9  frozenset({85123A})   frozenset({21733})  0.027980    0.221477  5.216862


# Compare Apriori vs FP-Growth Algorithms

In [89]:
import pandas as pd
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# ── Load basket matrix ─────────────────────────────────────────────────────
basket = pd.read_csv('basket_matrix.csv', index_col=0)
basket = basket.astype(bool)
print(f"Basket matrix loaded: {basket.shape}")

# ── Run Apriori ────────────────────────────────────────────────────────────
print("\n=== APRIORI ===")
start = time.time()
ap_itemsets = apriori(basket,
                      min_support=0.02,
                      use_colnames=True,
                      max_len=None)
end = time.time()
ap_time = round(end - start, 2)
print(f"Time taken:          {ap_time} seconds")
print(f"Frequent itemsets:   {len(ap_itemsets)}")

ap_rules = association_rules(ap_itemsets,
                              metric='lift',
                              min_threshold=1.0)
ap_rules['antecedents'] = ap_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
ap_rules['consequents'] = ap_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
ap_rules['support']     = ap_rules['support'].round(4)
ap_rules['confidence']  = ap_rules['confidence'].round(4)
ap_rules['lift']        = ap_rules['lift'].round(4)
print(f"Rules generated:     {len(ap_rules)}")

# ── Run FP-Growth ──────────────────────────────────────────────────────────
print("\n=== FP-GROWTH ===")
start = time.time()
fp_itemsets = fpgrowth(basket,
                       min_support=0.02,
                       use_colnames=True,
                       max_len=None)
end = time.time()
fp_time = round(end - start, 2)
print(f"Time taken:          {fp_time} seconds")
print(f"Frequent itemsets:   {len(fp_itemsets)}")

fp_rules = association_rules(fp_itemsets,
                              metric='lift',
                              min_threshold=1.0)
fp_rules['antecedents'] = fp_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
fp_rules['consequents'] = fp_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
fp_rules['support']     = fp_rules['support'].round(4)
fp_rules['confidence']  = fp_rules['confidence'].round(4)
fp_rules['lift']        = fp_rules['lift'].round(4)
print(f"Rules generated:     {len(fp_rules)}")

# ── Compare rules ──────────────────────────────────────────────────────────
print("\n=== COMPARISON ===")
merged = pd.merge(
    ap_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']],
    fp_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']],
    on=['antecedents', 'consequents'],
    how='outer',
    suffixes=('_apriori', '_fp'),
    indicator=True
)

both     = merged[merged['_merge'] == 'both']
only_ap  = merged[merged['_merge'] == 'left_only']
only_fp  = merged[merged['_merge'] == 'right_only']

print(f"Rules in both:            {len(both)}")
print(f"Rules only in Apriori:    {len(only_ap)}")
print(f"Rules only in FP-Growth:  {len(only_fp)}")

# ── Metric validation ──────────────────────────────────────────────────────
print("\n=== METRIC VALIDATION ===")
print(f"Support identical:    {(both['support_apriori']    == both['support_fp']).all()}")
print(f"Confidence identical: {(both['confidence_apriori'] == both['confidence_fp']).all()}")
print(f"Lift identical:       {(both['lift_apriori']       == both['lift_fp']).all()}")

# ── Final summary ──────────────────────────────────────────────────────────
print("\n=== FINAL SUMMARY ===")
print(f"{'Metric':<25} {'Apriori':>10} {'FP-Growth':>12}")
print("-" * 50)
print(f"{'Frequent itemsets':<25} {len(ap_itemsets):>10} {len(fp_itemsets):>12}")
print(f"{'Rules generated':<25} {len(ap_rules):>10} {len(fp_rules):>12}")
print(f"{'Execution time (s)':<25} {ap_time:>10} {fp_time:>12}")
print(f"{'Speed improvement':<25} {'baseline':>10} {str(round(ap_time/fp_time, 2))+'x faster':>12}")
print(f"{'Rules identical':<25} {'':>10} {'Yes':>12}")
print(f"{'Support identical':<25} {'':>10} {'Yes':>12}")
print(f"{'Confidence identical':<25} {'':>10} {'Yes':>12}")
print(f"{'Lift identical':<25} {'':>10} {'Yes':>12}")

Basket matrix loaded: (16512, 3890)

=== APRIORI ===
Time taken:          1.28 seconds
Frequent itemsets:   488
Rules generated:     312

=== FP-GROWTH ===
Time taken:          0.94 seconds
Frequent itemsets:   488
Rules generated:     312

=== COMPARISON ===
Rules in both:            312
Rules only in Apriori:    0
Rules only in FP-Growth:  0

=== METRIC VALIDATION ===
Support identical:    True
Confidence identical: True
Lift identical:       True

=== FINAL SUMMARY ===
Metric                       Apriori    FP-Growth
--------------------------------------------------
Frequent itemsets                488          488
Rules generated                  312          312
Execution time (s)              1.28         0.94
Speed improvement           baseline 1.36x faster
Rules identical                               Yes
Support identical                             Yes
Confidence identical                          Yes
Lift identical                                Yes


**Key Findings**

- Both algorithms produced exactly 312 rules with identical support, confidence, and lift values across every rule — confirming that the two methods are mathematically equivalent and the discovered associations are robust and algorithm-independent.
- However, FP-Growth completed the task in just 0.92 seconds compared to 1.29 seconds for Apriori, an 1.4x speed improvement on this dataset. This advantage arises because FP-Growth eliminates the candidate generation step that makes Apriori increasingly slow on high-dimensional datasets such as this basket matrix (16,512 × 3,890).
- Additionally, FP-Growth is likely to handle lower support thresholds, such as min_support = 0.01 — without the memory error encountered by Apriori, making it a more scalable choice for larger retail datasets.

**Algorithm Selection Justification**

The Apriori algorithm was selected as the primary method for this report because it is the most widely documented and cited approach in the association rule mining literature (Agrawal & Srikant, 1994), making it the most appropriate choice for an academic study. The FP-Growth comparison confirms that the rules produced are valid and reproducible regardless of the algorithm used.


#  Strongest Associations in the Dataset

Threshold:
- lift > 5  -  Our average lift was 7.60 — so lift > 5 captures above-average associations
- confidence > 0.6  -  Means the rule is correct more than 60% of the time — reliable enough to act on

The thresholds of lift > 5 and confidence > 0.6 were selected based on the distribution of the resulting rules. With an average lift of 7.60 across all 312 rules, a threshold of lift > 5 captures rules with stronger-than-average associations. A confidence threshold of 0.6 ensures that selected rules are correct in at least 60% of transactions, making them reliable enough for business decisions.

In [80]:
import pandas as pd

# Load resulting rules
resulting_rules = pd.read_csv('resulting_rules.csv')

print(f"Total rules: {len(resulting_rules)}")
print(f"Average lift: {resulting_rules['lift'].mean():.2f}")
print(f"Average confidence: {resulting_rules['confidence'].mean():.2f}")
print(f"Average support: {resulting_rules['support'].mean():.4f}")

# Filter strong rules from ALL 312 rules
strong_rules = resulting_rules[
    (resulting_rules['lift'] > 5) &
    (resulting_rules['confidence'] > 0.6)
].reset_index(drop=True)

# Save to CSV
strong_rules.to_csv('strong_rules.csv', index=False)

print(f"\nStrong rules (lift > 5 & confidence > 0.6): {len(strong_rules)}")
print("\n")
print(strong_rules[['antecedents',  'consequents', 'support','confidence','lift']].to_string())

Total rules: 312
Average lift: 7.60
Average confidence: 0.44
Average support: 0.0254

Strong rules (lift > 5 & confidence > 0.6): 47


     antecedents   consequents  support  confidence     lift
0         84596F        84596B   0.0203      0.8014  26.2046
1         84596B        84596F   0.0203      0.6634  26.2046
2          22577         22578   0.0223      0.7230  24.9748
3          22578         22577   0.0223      0.7699  24.9748
4   22699, 22697         22698   0.0298      0.7029  16.6269
5          22698  22699, 22697   0.0298      0.7049  16.6269
6          22866         22865   0.0200      0.6187  16.2673
7   22699, 22698         22697   0.0298      0.9028  15.9938
8   22698, 22697         22699   0.0298      0.8542  14.7996
9          22698         22697   0.0349      0.8252  14.6201
10         22697         22698   0.0349      0.6180  14.6201
11         22630         22629   0.0258      0.6094  14.3554
12         22629         22630   0.0258      0.6077  14.3554
13  22699, 

In [81]:
print(f"Strong Rules")
print(F"============")
print(f"Number of strong rules: {len(strong_rules)}")
print(f"Average lift: {strong_rules['lift'].mean():.2f}")
print(f"Average confidence: {strong_rules['confidence'].mean():.2f}")
print(f"Average support: {strong_rules['support'].mean():.4f}")
print(f"Highest lift: {strong_rules['lift'].max():.2f}")
print(f"Highest confidence: {strong_rules['confidence'].max():.4f}")

Strong Rules
Number of strong rules: 47
Average lift: 11.94
Average confidence: 0.69
Average support: 0.0278
Highest lift: 26.20
Highest confidence: 0.9028


## Full descriptions with StockCodes

In [84]:
import pandas as pd
import re

# Load strong rules
strong_rules = pd.read_csv('strong_rules.csv')

# Build description map from dataset
df_raw = pd.read_excel('Online Retail.xlsx')
df_raw['StockCode'] = df_raw['StockCode'].astype(str).str.strip()
desc_map = df_raw.dropna(subset=['Description'])\
                 .groupby('StockCode')['Description']\
                 .first()\
                 .to_dict()

# Extract all unique stock codes from antecedents and consequents
all_codes = set()
for val in strong_rules['antecedents'].tolist() + strong_rules['consequents'].tolist():
    for item in str(val).split(','):
        item = item.strip()
        if re.match(r'^\d+[A-Z]?$', item):
            all_codes.add(item)

print(f"Total unique stock codes found: {len(all_codes)}")
print("\n")

# Map each code to its description
code_desc = {code: desc_map.get(code, 'NOT FOUND') for code in sorted(all_codes)}

# Display as a clean dataframe
StockCode_description = pd.DataFrame(list(code_desc.items()), columns=['StockCode', 'Description'])
print(StockCode_description.to_string())

# Save to CSV
StockCode_description.to_csv('StockCode_description.csv', index=False)


Total unique stock codes found: 46


   StockCode                          Description
0      20711                      JUMBO BAG TOYS 
1      20712           JUMBO BAG WOODLAND ANIMALS
2      20713                       JUMBO BAG OWLS
3      20719               WOODLAND CHARLOTTE BAG
4      20723             STRAWBERRY CHARLOTTE BAG
5      20724          RED RETROSPOT CHARLOTTE BAG
6      20725              LUNCH BAG RED RETROSPOT
7      20727              LUNCH BAG  BLACK SKULL.
8      21733     RED HANGING HEART T-LIGHT HOLDER
9      21790                   VINTAGE SNAP CARDS
10     21791   VINTAGE HEADS AND TAILS CARD GAME 
11     21928       JUMBO BAG SCANDINAVIAN PAISLEY
12     21931               JUMBO STORAGE BAG SUKI
13     22086      PAPER CHAIN KIT 50'S CHRISTMAS 
14     22355            CHARLOTTE BAG SUKI DESIGN
15     22356          CHARLOTTE BAG PINK POLKADOT
16     22383              LUNCH BAG SUKI  DESIGN 
17     22384              LUNCH BAG PINK POLKADOT
18     22385 

# Recommendations

## Official Product Bundle Packages

In [92]:
import pandas as pd
import re

# ── Load data ──────────────────────────────────────────────────────────────
strong_rules = pd.read_csv('strong_rules.csv')
stock_desc   = pd.read_csv('StockCode_description.csv')

stock_desc['StockCode']   = stock_desc['StockCode'].astype(str).str.strip()
stock_desc['Description'] = stock_desc['Description'].astype(str).str.strip()
desc_map = dict(zip(stock_desc['StockCode'], stock_desc['Description']))

def resolve_desc(text):
    items = [i.strip() for i in str(text).split(',')]
    return ', '.join([desc_map.get(i, i) for i in items])

strong_rules['antecedent_desc'] = strong_rules['antecedents'].apply(resolve_desc)
strong_rules['consequent_desc'] = strong_rules['consequents'].apply(resolve_desc)
strong_rules['support']         = strong_rules['support'].round(4)
strong_rules['confidence']      = strong_rules['confidence'].round(4)
strong_rules['lift']            = strong_rules['lift'].round(4)

print("=" * 65)
print("BUNDLE RULE BASIS CALCULATIONS")
print("=" * 65)

# ── 1. Regency Tea Set — highest confidence ────────────────────────────────
print("\n1. REGENCY TEA SET")
teacup = strong_rules[
    strong_rules['antecedent_desc'].str.contains('TEACUP|REGENCY|CAKESTAND', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('TEACUP|REGENCY|CAKESTAND', case=False, na=False)
]
top_conf = teacup.loc[teacup['confidence'].idxmax()]
print(f"   Rule:       {top_conf['antecedent_desc']}")
print(f"            -> {top_conf['consequent_desc']}")
print(f"   Confidence: {top_conf['confidence']} = {top_conf['confidence']*100:.2f}%")
print(f"   Rule Basis: {round(top_conf['confidence']*100, 2)}% confidence")

# ── 2. Scandinavian Christmas Set — lift ───────────────────────────────────
print("\n2. SCANDINAVIAN CHRISTMAS SET")
christmas = strong_rules[
    strong_rules['antecedent_desc'].str.contains('WOODEN HEART|WOODEN STAR', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('WOODEN HEART|WOODEN STAR', case=False, na=False)
]
top_lift_xmas = christmas['lift'].max()
top_rule_xmas = christmas.loc[christmas['lift'].idxmax()]
print(f"   Rule:       {top_rule_xmas['antecedent_desc']}")
print(f"            -> {top_rule_xmas['consequent_desc']}")
print(f"   Lift:       {top_lift_xmas}")
print(f"   Rule Basis: Lift {top_lift_xmas}")

# ── 3. Jumbo Bag Collection — anchor count ─────────────────────────────────
print("\n3. JUMBO BAG COLLECTION")
jumbo = strong_rules[
    strong_rules['antecedent_desc'].str.contains('JUMBO|STORAGE BAG', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('JUMBO|STORAGE BAG', case=False, na=False)
]
total_jumbo = len(jumbo)
anchor_count = jumbo['consequent_desc'].str.contains(
    'JUMBO BAG RED RETROSPOT', case=False, na=False).sum()
print(f"   Total Jumbo Bag rules:              {total_jumbo}")
print(f"   Rules where RED RETROSPOT is cons.: {anchor_count}")
print(f"   Rule Basis: Anchors {anchor_count} of {total_jumbo} rules")

# ── 4. Decorative Bowl Set — highest lift ──────────────────────────────────
print("\n4. DECORATIVE BOWL SET")
bowls = strong_rules[
    strong_rules['antecedent_desc'].str.contains('BOWL|MARSHMALLOW|DOLLY MIX', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('BOWL|MARSHMALLOW|DOLLY MIX', case=False, na=False)
]
top_lift_bowl = bowls['lift'].max()
top_rule_bowl = bowls.loc[bowls['lift'].idxmax()]
print(f"   Rule:       {top_rule_bowl['antecedent_desc']}")
print(f"            -> {top_rule_bowl['consequent_desc']}")
print(f"   Lift:       {top_lift_bowl}")
print(f"   Rule Basis: Lift {top_lift_bowl}")

# ── 5. Alarm Clock Colour Set — lift range ─────────────────────────────────
print("\n5. ALARM CLOCK COLOUR SET")
clocks = strong_rules[
    strong_rules['antecedent_desc'].str.contains('ALARM CLOCK', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('ALARM CLOCK', case=False, na=False)
]
min_lift_clock = clocks['lift'].min()
max_lift_clock = clocks['lift'].max()
print(f"   Total Alarm Clock rules: {len(clocks)}")
print(f"   Lift range:              {min_lift_clock} – {max_lift_clock}")
print(clocks[['antecedent_desc', 'consequent_desc', 'lift']].to_string())
print(f"   Rule Basis: Lift {min_lift_clock}–{max_lift_clock}")

# ── 6. Hand Warmer Gift Set — lift ─────────────────────────────────────────
print("\n6. HAND WARMER GIFT SET")
warmers = strong_rules[
    strong_rules['antecedent_desc'].str.contains('HAND WARMER', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('HAND WARMER', case=False, na=False)
]
top_lift_warmer = warmers['lift'].max()
top_rule_warmer = warmers.loc[warmers['lift'].idxmax()]
print(f"   Rule:       {top_rule_warmer['antecedent_desc']}")
print(f"            -> {top_rule_warmer['consequent_desc']}")
print(f"   Lift:       {top_lift_warmer}")
print(f"   Rule Basis: Lift {top_lift_warmer}")

# ── 7. Garden Gift Set — higher confidence ─────────────────────────────────
print("\n7. GARDEN GIFT SET")
garden = strong_rules[
    strong_rules['antecedent_desc'].str.contains('GARDENERS|KNEELING', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('GARDENERS|KNEELING', case=False, na=False)
]
top_conf_garden = garden['confidence'].max()
top_rule_garden = garden.loc[garden['confidence'].idxmax()]
print(f"   Rule:       {top_rule_garden['antecedent_desc']}")
print(f"            -> {top_rule_garden['consequent_desc']}")
print(f"   Confidence: {top_conf_garden} = {top_conf_garden*100:.0f}%")
print(f"   Rule Basis: {round(top_conf_garden*100)}% confidence")

# ── 8. Vintage Games Pack — lift ───────────────────────────────────────────
print("\n8. VINTAGE GAMES PACK")
vintage = strong_rules[
    strong_rules['antecedent_desc'].str.contains('VINTAGE SNAP|VINTAGE HEADS', case=False, na=False) |
    strong_rules['consequent_desc'].str.contains('VINTAGE SNAP|VINTAGE HEADS', case=False, na=False)
]
top_lift_vintage = vintage['lift'].max()
top_rule_vintage = vintage.loc[vintage['lift'].idxmax()]
print(f"   Rule:       {top_rule_vintage['antecedent_desc']}")
print(f"            -> {top_rule_vintage['consequent_desc']}")
print(f"   Lift:       {top_lift_vintage}")
print(f"   Rule Basis: Lift {top_lift_vintage}")

# ── Final summary table ────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("FINAL BUNDLE SUMMARY TABLE")
print("=" * 65)

bundles = [
    ("Regency Tea Set",          f"{round(top_conf['confidence']*100, 2)}% confidence"),
    ("Scandinavian Christmas Set",f"Lift {top_lift_xmas}"),
    ("Jumbo Bag Collection",      f"Anchors {anchor_count} of {total_jumbo} rules"),
    ("Decorative Bowl Set",       f"Lift {top_lift_bowl}"),
    ("Alarm Clock Colour Set",    f"Lift {min_lift_clock}–{max_lift_clock}"),
    ("Hand Warmer Gift Set",      f"Lift {top_lift_warmer}"),
    ("Garden Gift Set",           f"{round(top_conf_garden*100)}% confidence"),
    ("Vintage Games Pack",        f"Lift {top_lift_vintage}"),
]

print(f"\n{'Bundle':<30} {'Rule Basis':<25}")
print("-" * 55)
for bundle, basis in bundles:
    print(f"{bundle:<30} {basis:<25}")

BUNDLE RULE BASIS CALCULATIONS

1. REGENCY TEA SET
   Rule:       ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY TEACUP AND SAUCER
            -> GREEN REGENCY TEACUP AND SAUCER
   Confidence: 0.9028 = 90.28%
   Rule Basis: 90.28% confidence

2. SCANDINAVIAN CHRISTMAS SET
   Rule:       WOODEN HEART CHRISTMAS SCANDINAVIAN
            -> WOODEN STAR CHRISTMAS SCANDINAVIAN
   Lift:       24.9748
   Rule Basis: Lift 24.9748

3. JUMBO BAG COLLECTION
   Total Jumbo Bag rules:              13
   Rules where RED RETROSPOT is cons.: 12
   Rule Basis: Anchors 12 of 13 rules

4. DECORATIVE BOWL SET
   Rule:       SMALL MARSHMALLOWS PINK BOWL
            -> SMALL DOLLY MIX DESIGN ORANGE BOWL
   Lift:       26.2046
   Rule Basis: Lift 26.2046

5. ALARM CLOCK COLOUR SET
   Total Alarm Clock rules: 3
   Lift range:              11.4888 – 11.645
               antecedent_desc             consequent_desc     lift
24  ALARM CLOCK BAKELIKE IVORY    ALARM CLOCK BAKELIKE RED  11.6450
26    ALARM CLOCK BAKE

## Implement a "Frequently Bought Together" Website Recommendation Engine

In [93]:
import pandas as pd

strong_rules = pd.read_csv('strong_rules.csv')

avg_lift   = strong_rules['lift'].mean()
sum_lift   = strong_rules['lift'].sum()
total      = len(strong_rules)

print(f"Total strong rules:  {total}")
print(f"Sum of all lifts:    {sum_lift:.4f}")
print(f"Average lift:        {sum_lift:.4f} / {total} = {avg_lift:.4f}")
print(f"Rounded to 2 dp:     {round(avg_lift, 2)}")

Total strong rules:  47
Sum of all lifts:    561.0996
Average lift:        561.0996 / 47 = 11.9383
Rounded to 2 dp:     11.94


The 47 strong rules have an average lift of 11.94, calculated as the arithmetic mean of lift values across all rules (sum of lifts: 561.10 ÷ 47 rules). This means that, on average, the co-purchase of associated products is nearly 12 times more likely than if the products were purchased independently, confirming that all 47 rules reflect genuine, commercially significant associations.